data/processed/
├── hospital_A/
│   ├── train.csv
│   └── val.csv
│
├── hospital_B/
│   ├── train.csv
│   └── val.csv
│
└── hospital_C/
    ├── train.csv
    └── val.csv

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [8]:
uci_clean = pd.read_csv("../data/processed/uci_clean.csv")
kaggle_clean = pd.read_csv("../data/processed/kaggle_clean.csv")

Assign Hospital-A (UCI CKD)

In [9]:
hospital_A = uci_clean.copy()

Split Kaggle Dataset → Hospital-B & C

We intentionally introduced label skew across hospitals to simulate real-world disease prevalence differences, resulting in a non-IID federated learning environment.

In [16]:
# Separate Kaggle data by class
ckd_data = kaggle_clean[kaggle_clean['classification'] == 1]
non_ckd_data = kaggle_clean[kaggle_clean['classification'] == 0]

# Hospital-B: CKD-heavy hospital (e.g., urban referral center)
hospital_B = pd.concat([
    ckd_data.sample(frac=0.7, random_state=42),
    non_ckd_data.sample(frac=0.3, random_state=42)
]).sample(frac=1, random_state=42)  # shuffle

# Hospital-C: remaining data (Non-CKD heavy)
hospital_C = kaggle_clean.drop(hospital_B.index)

Train/Validation Split Per Hospital

In [17]:
def local_split(df):
    return train_test_split(
        df,
        test_size=0.2,
        stratify=df['classification'],
        random_state=42
    )

A_train, A_val = local_split(hospital_A)
B_train, B_val = local_split(hospital_B)
C_train, C_val = local_split(hospital_C)


In [18]:
A_train.to_csv("../data/processed/hospital_A/train.csv", index=False)
A_val.to_csv("../data/processed/hospital_A/val.csv", index=False)

B_train.to_csv("../data/processed/hospital_B/train.csv", index=False)
B_val.to_csv("../data/processed/hospital_B/val.csv", index=False)
C_train.to_csv("../data/processed/hospital_C/train.csv", index=False)
C_val.to_csv("../data/processed/hospital_C/val.csv", index=False)

In [19]:
A_train.shape, A_val.shape, B_train.shape, B_val.shape, C_train.shape, C_val.shape

((320, 25), (80, 25), (176, 25), (44, 25), (144, 25), (36, 25))

Verify Non-IID Distribution

In [20]:
def show_dist(df, name):
    print(f"\n{name}")
    print(df['classification'].value_counts(normalize=True))

show_dist(A_train, "Hospital-A (UCI)")
show_dist(B_train, "Hospital-B (Kaggle)")
show_dist(C_train, "Hospital-C (Kaggle)")



Hospital-A (UCI)
classification
1    0.625
0    0.375
Name: proportion, dtype: float64

Hospital-B (Kaggle)
classification
1    0.795455
0    0.204545
Name: proportion, dtype: float64

Hospital-C (Kaggle)
classification
0    0.583333
1    0.416667
Name: proportion, dtype: float64
